# FHE-Flicker with TenSEAL (CKKS)

This notebook adapts the Flicker rebalancing framework to use **Fully Homomorphic Encryption**
via TenSEAL's CKKS scheme. Two operations are FHE-secured:

1. **Global Redundancy Check** — other clients' feature vectors are encrypted before being
   sent to the aggregator. The aggregator computes cosine similarities on ciphertexts
   and never sees raw feature data.

2. **Linear Classifier Inference** — at test time, the user encrypts their 512-dim feature
   vector. The server evaluates the trained linear head entirely in the encrypted domain
   and returns encrypted logits that only the user can decrypt.

Everything else (LD/GD accounting, dominant-client selection, oversampling, local redundancy)
stays in plaintext — those operations only touch class-count histograms (10 integers per
client), which carry no private feature information.

# **0. Imports**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.datasets as dset
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Subset, DataLoader, ConcatDataset
from collections import Counter
import tenseal as ts

# **1. CIFAR-10 + ResNet Feature Extractor (UNCHANGED)**

In [ ]:
transform_resnet = T.Compose([
    T.ToTensor(),
    T.Resize((224, 224), antialias=True),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

full_train_dataset = dset.CIFAR10(
    root="./data", train=True, download=True, transform=transform_resnet
)

all_labels = np.array(full_train_dataset.targets)

resnet = models.resnet18(weights="IMAGENET1K_V1")
resnet.fc = nn.Identity()
resnet.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = resnet.to(device)

# **2. Feature Extraction (UNCHANGED)**

In [ ]:
def extract_features_batched(dataset, batch_size=256):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    feats = []
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            f = resnet(x)
            feats.append(f.cpu().numpy())
    return np.concatenate(feats, axis=0)

# **3. TenSEAL CKKS Context Setup (NEW)**

In [ ]:
# --- CKKS Parameter Rationale ---
#
# poly_modulus_degree = 8192
#   The ring dimension N. Must be a power of 2.
#   Gives N/2 = 4096 CKKS plaintext slots (each slot holds one float).
#   Our 512-dim feature vectors use 512 slots, well within the 4096 limit.
#   At N=8192 with the modulus chain below, security >= 128 bits (per HE-Std).
#   N=4096 would be too small (insecure with 4 primes); N=16384 is overkill
#   for a depth-2 circuit and would be 4x slower.
#
# coeff_mod_bit_sizes = [60, 40, 40, 60]
#   The modulus chain (RNS primes in bits). Sum = 200 bits.
#   - First prime (60 bits): the 'special' prime for key switching.
#   - Two middle primes (40 bits each): one consumed per multiplication level.
#     Two middle primes => supports multiplicative depth 2, enough for:
#       level 1: dot product (one multiply+rotate+sum)
#       level 2: bias addition (plaintext, costs no level) + any future op
#   - Last prime (60 bits): the 'special' prime for relinearisation keys.
#   If we added polynomial activations (e.g., approx ReLU), we'd need more
#   40-bit primes, which would force N up to 16384 to stay >= 128-bit secure.
#
# global_scale = 2**40
#   CKKS encodes floats as integers scaled by 2^scale_bits.
#   2^40 gives ~12 decimal digits of precision before quantisation noise
#   becomes significant. This matches the 40-bit middle primes so that each
#   rescale (after a multiply) consumes exactly one level cleanly.
#   2^30 would be faster but too noisy for 512-dim dot products.
#   2^50 would be more precise but exceed the 40-bit prime budget.

def setup_ckks_context():
    ctx = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60]
    )
    # Galois keys enable the rotation operations used internally by
    # ckks_vector.dot() to sum up the element-wise products into a scalar.
    # Without them, .dot() would raise a runtime error.
    ctx.generate_galois_keys()
    ctx.global_scale = 2**40
    return ctx

ctx = setup_ckks_context()
print("TenSEAL CKKS context ready.")
print(f"  poly_modulus_degree : 8192")
print(f"  coeff_mod_bit_sizes : [60, 40, 40, 60]  (depth-2 circuit)")
print(f"  global_scale        : 2^40  (~12 decimal digits of precision)")
print(f"  security level      : >= 128-bit (HE-Standard)")

## 3b. Encryption Helpers

Features are **L2-normalised on the client before encryption**.
This is critical: CKKS has no native square-root, so computing the norm
inside the encrypted circuit would require a polynomial approximation
and consume extra multiplicative levels. Pre-normalising in plaintext
is free and does not weaken the privacy model — the server only learns
the direction of the feature vector, not its magnitude.

In [ ]:
def l2_normalize(feats):
    """L2-normalise a (N, D) feature matrix row-wise (done client-side, plaintext)."""
    norms = np.linalg.norm(feats, axis=1, keepdims=True)
    return feats / np.maximum(norms, 1e-9)


def encrypt_feature_matrix(ctx, feats_np):
    """
    feats_np : (N, 512) float32 array, already L2-normalised.
    Returns  : list of N ts.CKKSVector ciphertexts.

    Each ciphertext encrypts one 512-dim feature vector into 512 of the
    4096 available CKKS slots.  The remaining 3584 slots are zero-padded
    (TenSEAL handles this automatically).
    """
    return [ts.ckks_vector(ctx, feat.tolist()) for feat in feats_np]


def decrypt_scalar(enc_result):
    """
    Decrypt a CKKS dot-product result.
    After .dot(), TenSEAL places the sum in slot 0 (and duplicates it
    across slots via rotation-and-sum); we read slot 0.
    """
    return enc_result.decrypt()[0]

# **4. Imbalanced Data Distribution (UNCHANGED)**

In [ ]:
def generate_imbalanced_split(total, n_clients, alpha=0.5):
    proportions = np.random.dirichlet(alpha=[alpha] * n_clients)
    raw_counts = (proportions * total).astype(int)
    diff = total - raw_counts.sum()
    raw_counts[np.argmax(raw_counts)] += diff
    return raw_counts


def rotate_list(lst, k):
    return lst[k:] + lst[:k]


def distribute_cifar_imbalanced(labels, n_clients, alpha=0.5):
    client_indices = {i: [] for i in range(n_clients)}
    for cls in range(10):
        cls_idx = np.where(labels == cls)[0]
        np.random.shuffle(cls_idx)
        base_split = generate_imbalanced_split(
            total=len(cls_idx), n_clients=n_clients, alpha=alpha
        )
        split = rotate_list(list(base_split), cls % n_clients)
        print(f"cls: {cls}  split: {split}")
        start = 0
        for cid in range(n_clients):
            count = split[cid]
            client_indices[cid].extend(cls_idx[start:start + count])
            start += count
        assert start == len(cls_idx)
    return client_indices

# **5. Local & Global Distributions and Normalization (UNCHANGED)**

In [ ]:
def compute_LIn(y):
    cnt = np.bincount(y, minlength=10)
    return cnt.min() / cnt.max()

# **6. Dominant Client Selection (UNCHANGED)**

In [ ]:
# Dominant client selection operates entirely on 10-dim class-count
# histograms (LD/GD). These are plaintext integers — no private feature
# data is involved, so no FHE is needed here.

def find_dominant_client(LDs_norm, GD_norm, excluded_clients):
    """
    LDs_norm : dict {cid -> L2-normalised 10-dim LD vector}
    GD_norm  : L2-normalised 10-dim GD vector
    Returns the client whose LD has the highest cosine similarity with GD.
    """
    sims = {}
    for cid, ld in LDs_norm.items():
        if cid in excluded_clients:
            continue
        sims[cid] = float(np.dot(ld, GD_norm))  # both already L2-normalised
    if not sims:
        return None
    return max(sims, key=sims.get)

# **7. Flicker Oversampling (UNCHANGED)**

In [ ]:
# Oversampling is purely local: the dominant client augments its own minority
# classes with random flips/rotations.  No inter-client communication happens,
# so no FHE is required.

def flicker_oversample(X_client, y_client, lin_thres=0.5):
    class_counts = Counter(y_client)
    Lmax = max(class_counts.values())
    new_X = [X_client]
    new_y = list(y_client)

    aug = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomRotation(10),
        T.ToTensor(),
        T.Resize((224, 224)),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    for cls, cnt in class_counts.items():
        Lin = cnt / Lmax
        if Lin < lin_thres:
            N_add = int(np.ceil(1 / Lin))
            cls_idx = np.where(y_client == cls)[0]
            chosen = np.random.choice(cls_idx, N_add, replace=True)
            base_ds = dset.CIFAR10(root="./data", train=True, download=False, transform=None)
            imgs, labels = [], []
            for idx in chosen:
                img, _ = base_ds[X_client.indices[idx]]
                imgs.append(aug(img))
                labels.append(cls)
            new_X.append(torch.utils.data.TensorDataset(
                torch.stack(imgs), torch.tensor(labels)
            ))
            new_y.extend(labels)

    return ConcatDataset(new_X), np.array(new_y)

# **8. Flicker Undersampling — LOCAL Redundancy (UNCHANGED)**

In [ ]:
# Local redundancy operates only on the dominant client's own data.
# No cross-client information is exchanged, so no FHE needed.

def get_majority_classes(y, lin_threshold=0.9, n_classes=10):
    cnt = np.bincount(y, minlength=n_classes)
    Lmax = cnt.max()
    return [c for c in range(n_classes) if cnt[c] / Lmax >= lin_threshold]


def local_redundancy_majority(X, y, theta, lin_threshold=0.9):
    """
    Returns (buffer_idx, buffer_vecs):
      buffer_idx  : indices into X of the top-theta redundant majority samples
      buffer_vecs : their L2-normalised feature vectors (512-dim, float32)
    """
    maj_classes = get_majority_classes(y, lin_threshold)
    if not maj_classes:
        return np.array([]), np.array([])

    maj_idx = np.where(np.isin(y, maj_classes))[0]
    X_maj = Subset(X, maj_idx)

    feats = extract_features_batched(X_maj)
    feats = l2_normalize(feats)  # normalise once; reused in FHE step

    sim = feats @ feats.T
    mean_sim = sim.mean(axis=1)
    var_sim = sim.var(axis=1)

    k = max(1, int(theta * len(mean_sim)))
    buffer_local = np.argsort(mean_sim)[-k:]
    buffer_local = buffer_local[np.argsort(var_sim[buffer_local])[::-1]]

    return maj_idx[buffer_local], feats[buffer_local]

# **9. Flicker Undersampling — FHE Global Redundancy (NEW)**

This is the main FHE operation.  The privacy model is:

- **Dominant client** publishes its buffer vectors as plaintext queries
  (these are candidate-for-removal samples, not the full dataset).
- **Other clients** encrypt their feature vectors locally before uploading.
  The aggregator (server) never sees their raw features.
- **Aggregator** evaluates `enc_feat · buf_vec` — a CKKS ciphertext × plaintext
  dot product — and returns encrypted similarity scores.
- The encrypted scores are decrypted (by the key holder) to make the
  remove/keep decision.

Because both vectors are L2-normalised before encryption, the dot product
equals the cosine similarity exactly, without any division inside FHE.

**FHE_SAMPLE_SIZE** caps how many other-client vectors are used per round.
In a real deployment each client would upload all their encrypted features;
here we sample to keep notebook runtime tractable.

In [ ]:
# FHE_SAMPLE_SIZE: number of encrypted feature vectors sampled from each
# non-dominant client for the global redundancy check.
#
# Why 200?
#   - Each TenSEAL CKKS dot product (512-dim, depth-1) takes ~5-15 ms on CPU.
#   - With 3 clients, 2 non-dominant, 200 samples each, ~20 buffer vectors:
#     20 * 2 * 200 = 8000 dot products  ~  40-120 seconds per round.
#   - Increasing to 1000 would give a better estimate of global redundancy
#     but would take 5-10 min per round. 200 is the notebook-friendly sweet spot.
#   - In production (GPU-accelerated or batched CKKS), the full dataset is used.

FHE_SAMPLE_SIZE = 200

In [ ]:
def fhe_undersample_global(
    ctx, dom_id, clients, theta=0.2, eta=0.6, lin_threshold=0.9
):
    """
    FHE-secured global redundancy undersampling.

    Parameters
    ----------
    ctx          : TenSEAL CKKS context (holds public/secret keys for simulation)
    dom_id       : ID of the dominant client
    clients      : dict {cid -> {"X": Subset, "y": np.array}}
    theta        : fraction of majority samples to buffer locally (0.2 = 20%)
    eta          : cosine similarity threshold above which a buffer sample
                   is considered globally redundant and removed (0.6)
                   Same value as plaintext version — semantics unchanged because
                   encrypted dot product == cosine sim after pre-normalisation.
    lin_threshold: a class is "majority" if its count / max_count >= this (0.9)

    Returns
    -------
    (Subset, np.array) : pruned X and y for the dominant client
    """
    Xd = clients[dom_id]["X"]
    yd = clients[dom_id]["y"]

    # --- Step 1: Local redundancy candidates (plaintext, dominant client only) ---
    buffer_idx, buffer_vecs = local_redundancy_majority(Xd, yd, theta, lin_threshold)
    if len(buffer_idx) == 0:
        print("  No majority-class buffer candidates found. Skipping undersampling.")
        return Xd, yd

    print(f"  Local buffer size: {len(buffer_idx)} candidates from majority classes")

    # --- Step 2: Encrypt other clients' features (simulates client-side encryption) ---
    # In a real system each client would encrypt locally and upload ciphertexts.
    # Here we simulate that: extract plaintext features, normalise, then encrypt.
    other_enc_feats = {}  # {cid: list[CKKSVector]}

    for cid, client in clients.items():
        if cid == dom_id:
            continue

        feats = extract_features_batched(client["X"])     # plaintext, client-side
        feats = l2_normalize(feats)                        # normalise before encrypting

        # Sample FHE_SAMPLE_SIZE vectors for tractable demo
        sample_idx = np.random.choice(
            len(feats), min(FHE_SAMPLE_SIZE, len(feats)), replace=False
        )
        feats_sample = feats[sample_idx]

        print(f"  Encrypting {len(feats_sample)} feature vectors from client {cid}...")
        other_enc_feats[cid] = encrypt_feature_matrix(ctx, feats_sample)

    # --- Step 3: FHE dot products (aggregator side) ---
    # For each buffer vector (plaintext query), compute its cosine similarity
    # against each encrypted feature from other clients.
    # enc_feat.dot(buf_vec.tolist()) is a CKKS ciphertext × plaintext inner product:
    #   - Server knows buf_vec (dominant client published it)
    #   - Server never sees enc_feat in plaintext
    #   - Result is an encrypted scalar; only the key holder can decrypt it

    remove = []

    for i, (buf_idx, buf_vec) in enumerate(zip(buffer_idx, buffer_vecs)):
        client_avgs = []

        for cid, enc_feats in other_enc_feats.items():
            sims = []
            for enc_f in enc_feats:
                enc_sim = enc_f.dot(buf_vec.tolist())   # FHE dot product
                sim_val = decrypt_scalar(enc_sim)       # decrypt -> float
                sims.append(sim_val)
            client_avgs.append(float(np.mean(sims)))

        final_avg = float(np.mean(client_avgs))

        # Decision made in plaintext on the decrypted scalar
        if final_avg >= eta:
            remove.append(buf_idx)

    print(f"  Removing {len(remove)} globally redundant samples")

    # --- Step 4: Apply mask ---
    mask = np.ones(len(Xd), dtype=bool)
    mask[remove] = False
    return Subset(Xd, np.where(mask)[0]), yd[mask]

# **10. Full Flicker Loop**

In [ ]:
MAX_DOM_ROUNDS = 6
LIn_MAX = 0.75

In [ ]:
def safe_collate(batch):
    xs, ys = zip(*batch)
    return torch.stack(xs), torch.tensor(ys, dtype=torch.long)

In [ ]:
def run_fhe_flicker(ctx, alpha=0.5, seed=None, n_clients=3, rounds=25):
    """
    Full FHE-Flicker experiment.

    The only structural difference from the plaintext version is that
    the global undersampling step (Step 5b) calls fhe_undersample_global()
    instead of flicker_undersample_global(), injecting encrypted dot products
    for the cross-client similarity check.
    """
    print(f"\n{'='*50}")
    print(f"FHE-Flicker  |  alpha={alpha}  |  seed={seed}")
    print(f"{'='*50}\n")

    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)

    # 1. Create imbalanced clients
    client_indices = distribute_cifar_imbalanced(
        labels=all_labels, n_clients=n_clients, alpha=alpha
    )
    clients = {}
    for cid in range(n_clients):
        Xc = Subset(full_train_dataset, client_indices[cid])
        yc = all_labels[client_indices[cid]]
        clients[cid] = {"X": Xc, "y": yc}
        print(f"Client {cid} class dist: {Counter(yc)}")

    # 2. Bookkeeping
    dominance_count = {cid: 0 for cid in clients}
    excluded_clients = set()
    client_resample_flag = {cid: 0 for cid in clients}  # 0=oversample, 1=undersample

    # 3. Flicker rounds
    for r in range(rounds):
        print(f"\n------ ROUND {r} ------")

        LDs = {
            cid: np.bincount(clients[cid]["y"], minlength=10)
            for cid in clients
        }
        LDs_norm = {
            cid: ld / np.linalg.norm(ld)
            for cid, ld in LDs.items()
        }
        GD = sum(LDs.values())
        GD_norm = GD / np.linalg.norm(GD)

        dom = find_dominant_client(LDs_norm, GD_norm, excluded_clients)
        if dom is None:
            print("No eligible dominant clients. Stopping.")
            break

        print(f"Dominant client: {dom}")
        Lin_dom = compute_LIn(clients[dom]["y"])

        if dominance_count[dom] >= MAX_DOM_ROUNDS or Lin_dom >= LIn_MAX:
            print(f"Client {dom} saturated (rounds={dominance_count[dom]}, LIn={Lin_dom:.3f})")
            excluded_clients.add(dom)
            continue

        if client_resample_flag[dom] == 0:
            print("-> Oversampling (plaintext, local)")
            Xn, yn = flicker_oversample(clients[dom]["X"], clients[dom]["y"])
            client_resample_flag[dom] = 1
        else:
            print("-> Undersampling (Local plaintext + FHE global)")
            Xn, yn = fhe_undersample_global(ctx, dom, clients)
            client_resample_flag[dom] = 0

        clients[dom]["X"] = Xn
        clients[dom]["y"] = yn
        dominance_count[dom] += 1
        print(f"Updated dist: {Counter(yn)}")

    # 4. Merge
    final_dataset = ConcatDataset([clients[cid]["X"] for cid in clients])
    final_labels = np.concatenate([clients[cid]["y"] for cid in clients])
    print(f"\nFinal size: {len(final_dataset)}")
    print(f"Final class dist: {Counter(final_labels)}")

    return final_dataset, final_labels

# **11. Train Model**

In [ ]:
test_dataset = dset.CIFAR10(
    root="./data", train=False, download=True, transform=transform_resnet
)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
# The linear head (nn.Linear(512, 10)) is the ideal FHE classifier:
#   y = W * x + b  is a single matrix-vector multiply + bias — depth-1 circuit.
# No activation function is applied inside the encrypted domain.
# Argmax is computed in plaintext after decryption.

class CIFARClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone.fc = nn.Identity()
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.head = nn.Linear(512, 10)

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)

In [ ]:
def train(model, loader, optimizer, criterion, epochs=10):
    for ep in range(epochs):
        model.train()
        total_loss = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)
        print(f"Epoch {ep}: loss = {total_loss / len(loader.dataset):.4f}")


def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

In [ ]:
# Run one full FHE-Flicker experiment
final_dataset, final_labels = run_fhe_flicker(ctx, alpha=0.5, seed=42)

In [ ]:
train_loader = DataLoader(
    final_dataset, batch_size=128, shuffle=True,
    num_workers=2, pin_memory=True, collate_fn=safe_collate
)

model = CIFARClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-3)

train(model, train_loader, optimizer, criterion, epochs=10)

In [ ]:
plain_acc = evaluate(model, test_loader)
print(f"Plaintext test accuracy: {plain_acc:.4f}")

# **12. FHE Linear Inference (NEW)**

At inference time the user does NOT want to reveal their image features to the server.
The protocol:

1. User extracts a 512-dim feature vector locally (ResNet18 runs on-device, plaintext).
2. User encrypts the feature vector with TenSEAL CKKS and sends the ciphertext.
3. Server holds the trained weight matrix W (10×512) and bias b (10,) in plaintext.
4. Server evaluates `enc_logit_i = enc_feat · W[i] + b[i]` for each class i — a
   CKKS ciphertext × plaintext dot product, followed by a plaintext scalar addition.
5. Server sends 10 encrypted logits back. User decrypts and takes argmax.

The server learns nothing about the user's feature vector beyond the final class
prediction (which is also encrypted until the user decrypts it).

**Note on `n_fhe_samples`:** full-testset FHE inference (10 000 samples × 10 dot products
each) would take ~30-60 min on CPU. We run on a small sample and compare against
plaintext predictions on the same samples to verify correctness.

In [ ]:
def fhe_linear_inference(ctx, model, test_loader, n_fhe_samples=100):
    """
    Encrypted inference using the trained linear head.

    Parameters
    ----------
    ctx            : TenSEAL CKKS context
    model          : trained CIFARClassifier
    test_loader    : DataLoader for the test set
    n_fhe_samples  : number of test samples to run under encryption
                     (100 takes ~30-60 s on CPU; increase for more rigorous testing)

    Returns
    -------
    fhe_preds   : np.array of predicted class indices (via FHE)
    plain_preds : np.array of predicted class indices (via plaintext, same samples)
    true_labels : np.array of ground-truth labels
    """
    # Extract weight matrix and bias from the trained head (plaintext, on server)
    W = model.head.weight.detach().cpu().numpy()   # (10, 512)
    b = model.head.bias.detach().cpu().numpy()     # (10,)

    fhe_preds   = []
    plain_preds = []
    true_labels = []
    count = 0

    model.eval()

    for x, y in test_loader:
        if count >= n_fhe_samples:
            break

        with torch.no_grad():
            # Plaintext feature extraction (client-side, ResNet backbone)
            feats = model.backbone(x.to(device)).cpu().numpy()  # (B, 512)

            # Plaintext predictions for comparison
            plain_out = model.head(torch.tensor(feats)).argmax(dim=1).numpy()

        for feat, p_pred, label in zip(feats, plain_out, y.numpy()):
            if count >= n_fhe_samples:
                break

            # --- Client side: encrypt feature vector ---
            enc_feat = ts.ckks_vector(ctx, feat.tolist())

            # --- Server side: evaluate encrypted linear head ---
            # For each of the 10 classes:
            #   enc_logit_i = enc_feat · W[i]  (CKKS × plaintext dot product)
            #   then add plaintext bias b[i]
            logits = []
            for i in range(10):
                enc_dot  = enc_feat.dot(W[i].tolist())    # encrypted dot product
                enc_logit = enc_dot + b[i].item()         # add bias in encrypted domain
                logit_val = decrypt_scalar(enc_logit)     # client decrypts
                logits.append(logit_val)

            fhe_preds.append(int(np.argmax(logits)))
            plain_preds.append(int(p_pred))
            true_labels.append(int(label))
            count += 1

    return np.array(fhe_preds), np.array(plain_preds), np.array(true_labels)

In [ ]:
print("Running FHE inference on 100 test samples (CKKS encrypted dot products)...")
fhe_preds, plain_preds, true_labels = fhe_linear_inference(
    ctx, model, test_loader, n_fhe_samples=100
)

# **13. Results: FHE vs Plaintext Accuracy**

In [ ]:
fhe_acc   = np.mean(fhe_preds   == true_labels)
plain_acc_sample = np.mean(plain_preds == true_labels)
agreement = np.mean(fhe_preds   == plain_preds)

print(f"Plaintext accuracy (sample)  : {plain_acc_sample:.4f}")
print(f"FHE accuracy (sample)        : {fhe_acc:.4f}")
print(f"FHE / Plaintext agreement    : {agreement:.4f}")
print()
print("Note: FHE accuracy should match plaintext accuracy to within CKKS noise")
print("(typically < 0.5% deviation for a depth-1 linear layer at scale=2^40).")

In [ ]:
# Optional: inspect the CKKS approximation error on one sample
import torch

model.eval()
x_sample, _ = next(iter(test_loader))
with torch.no_grad():
    feat = model.backbone(x_sample[:1].to(device)).cpu().numpy()[0]  # (512,)

W = model.head.weight.detach().cpu().numpy()
b = model.head.bias.detach().cpu().numpy()

# Plaintext logits
plain_logits = W @ feat + b

# FHE logits
enc_feat = ts.ckks_vector(ctx, feat.tolist())
fhe_logits = []
for i in range(10):
    enc_logit = enc_feat.dot(W[i].tolist()) + b[i].item()
    fhe_logits.append(decrypt_scalar(enc_logit))
fhe_logits = np.array(fhe_logits)

print("Plaintext logits :", np.round(plain_logits, 4))
print("FHE logits       :", np.round(fhe_logits,   4))
print("Max abs error    :", np.max(np.abs(plain_logits - fhe_logits)))
print("Plaintext pred   :", np.argmax(plain_logits))
print("FHE pred         :", np.argmax(fhe_logits))